### Problem 1 (SJ3.33)
##### Integrate MRP's, ensuring switching condition prevents divergence. 

In [5]:
from attitude_package.dcms import Attitude,Linear
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
at = Attitude()

def omega(t):
    """   
    omega as a function of time
    """
    omega = np.array([1,0.5,-0.7]) # rad/s
    return omega

# Required initial parameters and values
sigma0 = np.array([0,0,0])
tspan = [0,5]
dt = 1e-3

# Performing integration
t,s = at.integrate_mrp(omega,sigma0,tspan,dt)
s1 = [si[0] for si in s]
s2 = [si[1] for si in s]
s3 = [si[2] for si in s]

# Plotting
fig = go.Figure(data = [go.Scatter(x=t,y=s1,mode = 'lines',name='sigma(1)'),
                        go.Scatter(x=t,y=s2,mode = 'lines',name='sigma(2)'),
                        go.Scatter(x=t,y=s3,mode = 'lines',name='sigma(3)')],
                        layout_title_text="Integrated MRP")
fig.update_xaxes(title_text = 'Time (s)')
fig.update_yaxes(title_text='MRP Values')
fig.show()

### Problem 4
#### Simulating a Rocket's trajectory

In [27]:
from attitude_package.dcms import Attitude,Linear
from attitude_package.numerical import Integrator
import numpy as np
import plotly.graph_objects as go

# Initializing classes for analysis
lin = Linear()
at = Attitude()
integrate = Integrator()

# Equation of motion function
def rocket_eom(t,x:list,c:dict)->list:
    """ 
    Eqution of motion for rocket as described in problem 4 of HW3

    Args:
        t: time
        x: state vector (x,y,z,vx,vy,vz,alpa,beta,gamma,omega1,omega2,omega3)
        c: Constant dict containing givens
    
    Returns:
        xdot: derivative of the state vector for integration
    """

    # Finding Thrust vector in body frame
    if t<10:
        T_b = np.array([c['T']*np.sin(c['xi']),
                        0,
                        c['T']*np.cos(c['xi'])]) # Body frame thrust
    else:
        T_b = np.array([0,0,0])

    # Finding torque due to T
    rT_b = np.array([0,0,-c['rnozz']])                              # Location of nozzle thrust in body frame
    L_b = np.cross(rT_b,T_b)                                          # Torque in body frame

    # Finding omega and omega_dot
    omega = np.array([x[-3],x[-2],x[-1]])
    omegad = -np.linalg.inv(c['Ic']) @ lin.tilde(omega) @ c['Ic']@omega + -np.linalg.inv(c['Ic']) @ L_b  # All I,L,omega in body frame
    
    # Finding new Euler angle rates
    alpha,beta,gamma = x[6],x[7],x[8]
    BN = at.dcm_123(alpha,beta,gamma)
    alphad,betad,gammad = at.ea_rate_123(alpha,beta,gamma,omega)

    
    # Finding translational derivatives
    T_n = np.linalg.inv(BN)@T_b 
    Fg_n = np.array([0,0,-c['g']])
    Fnet_n = T_n + Fg_n
    xdd = 1/c['m']*Fnet_n[0]
    ydd = 1/c['m']*Fnet_n[1]
    zdd = 1/c['m']*Fnet_n[2]

    xd = x[3]
    yd = x[4]
    zd = x[5]
    # Forming the state vector derivative
    xdot = np.array([xd,yd,zd,xdd,ydd,zdd,alphad,betad,gammad,omegad[0],omegad[1],omegad[2]])

    return xdot

# Function to  find the location of the nosecone

# Givens and time span
givens = {'m': 1,'xi': np.deg2rad(2.5),'T':15,'g':9.81,'Ic':np.array([[32.5,0,0],[0,32.5,0],[0,0,5]]),'rnozz':3,'rnose':4,'twopi':True}
tspan = [0,60]

# Initial attitude determination
BN0 = np.array([[0,-1,0],
                [1,0,0],
                [0,0,1]])
a0,b0,g0 = at.inv_123(BN0)

# Initial spin of rocket
omega0 = np.array([0,0,0.85])

# Initial State Vector
x0 = np.array([0,0,givens['rnozz'],0,0,0,a0,b0,g0,omega0[0],omega0[1],omega0[2]])

# Performing integration
t,x = integrate.ode45(rocket_eom,x0,tspan,other=givens)

# Converting angles to be between 0,2pi
alpha = x[6]
beta = x[7]
gamma = x[8]
if givens['twopi']:
    alpha = [a % (2*np.pi) for a in alpha]
    beta = [b % (2*np.pi) for b in beta]
    gamma = [g % (2*np.pi) for g in gamma]

# Creating position vectors and such
r = [[x[0][i],x[1][i],x[2][i]] for i in range(len(t))]

ground_index = next(i for i, z in enumerate(x[2]) if z<0)
r_ground = [[x[0][i],x[1][i],x[2][i]] for i in range(ground_index)]

# Plotting
############################### OMEGA PLOT ###############################
fig = go.Figure(data = [go.Scatter(x=t,y=x[9],mode = 'lines',name='w1'),
                        go.Scatter(x=t,y=x[10],mode = 'lines',name='w2'),
                        go.Scatter(x=t,y=x[11],mode = 'lines',name='w3')],
                        layout_title_text="Integrated Angular Velocity in Principal Frame")
fig.update_xaxes(title_text = 'Time (s)')
fig.update_yaxes(title_text='Omega (rad/s)')
fig.show()
############################### VELOCITY PLOT (NO GROUND) ###############################
fig = go.Figure(data = [go.Scatter(x=t,y=x[3],mode = 'lines',name='vx'),
                        go.Scatter(x=t,y=x[4],mode = 'lines',name='vy'),
                        go.Scatter(x=t,y=x[5],mode = 'lines',name='vz')],
                        layout_title_text="Integrated Velocity with no Ground Enforced")
fig.update_xaxes(title_text = 'Time (s)')
fig.update_yaxes(title_text='Velocity (m/s)')
fig.show()
############################### VELOCITY PLOT (GROUND) ###############################
fig = go.Figure(data = [go.Scatter(x=t[:ground_index],y=x[3][:ground_index],mode = 'lines',name='vx'),
                        go.Scatter(x=t[:ground_index],y=x[4][:ground_index],mode = 'lines',name='vy'),
                        go.Scatter(x=t[:ground_index],y=x[5][:ground_index],mode = 'lines',name='vz')],
                        layout_title_text="Integrated Velocity with Ground Enforced")
fig.update_xaxes(title_text = 'Time (s)')
fig.update_yaxes(title_text='Velocity (m/s)')
fig.show()
############################### EULER ANGLE PLOT ###############################
fig = go.Figure(data = [go.Scatter(x=t,y=np.rad2deg(alpha),mode = 'lines',name='alpha'),
                        go.Scatter(x=t,y=np.rad2deg(beta),mode = 'lines',name='beta')],
                        layout_title_text="Integrated Euler Angles")
fig.update_xaxes(title_text = 'Time (s)')
fig.update_yaxes(title_text='Euler Angles (deg)')
fig.show()
############################### 3D PLOT (CG) (GROUND) ###############################
trace = go.Scatter3d(
    x=[ri[0] for ri in r_ground],
    y=[ri[1] for ri in r_ground],
    z=[ri[2] for ri in r_ground],
    mode='lines+markers', 
    marker=dict(
        size=1,
        color=t, 
        colorscale='Viridis',
        opacity=0.8
    )
)

fig = go.Figure(data=[trace])
fig.update_layout(
    title=dict(
        text='3D Position of Rocket (Ground enforced)',
        x=0.5,
        xanchor='center'
    ),
    scene=dict( 
        xaxis_title='x (m)',
        yaxis_title='y (m)',
        zaxis_title='z (m)'
    ),
    margin=dict(
        l=0,
        r=0,
        t=95,
        b=0,
        pad=0
    )
)

fig.show()
############################### 3D PLOT (CG) (NO GROUND) ###############################
trace = go.Scatter3d(
    x=[ri[0] for ri in r],
    y=[ri[1] for ri in r],
    z=[ri[2] for ri in r],
    mode='lines+markers', 
    marker=dict(
        size=1,
        color=t, 
        colorscale='Viridis',
        opacity=0.8
    )
)

fig = go.Figure(data=[trace])
fig.update_layout(
    title=dict(
        text='3D Position of Rocket (No Ground enforced)',
        x=0.5,
        xanchor='center'
    ),
    scene=dict( 
        xaxis_title='x (m)',
        yaxis_title='y (m)',
        zaxis_title='z (m)'
    ),
    margin=dict(
        l=0,
        r=0,
        t=95,
        b=0,
        pad=0
    )
)

fig.show()

############################### 3D PLOT (NOSE) (GROUND) ###############################
def nose_cone(x:list,c:dict)->list:
    """ 
    Iterate through the state vector and use the geometric paramters to calculate the location of the nose cone

    Args:
        x: state vector solution after integrating
        c: givens dict containing r_nose
    """
    rnose = c['rnose']
    r_nose = []

    for xi in x.T:
        a,b,g = xi[6],xi[7],xi[8]
        BN = at.dcm_123(a,b,g)
        r_cg = [xi[0],xi[1],xi[2]]
        rn_cg_N = BN@np.array([0,0,rnose])
        r_nose.append(rn_cg_N+r_cg)
    return r_nose

r_nose = nose_cone(x,givens)
fig = go.Figure(data = [go.Scatter(x=r_nose[0][:ground_index],y=r_nose[2][:ground_index],mode = 'lines',name='')],
                        layout_title_text="x-y of Nose Cone in Inertial Coordinates")
fig.update_xaxes(title_text = 'x (m)')
fig.update_yaxes(title_text='y (m)')
fig.show()